# Script 20: Hierarchical KAN Architecture (Unified Git & Zip Deployment)
This notebook clones the latest codebase from GitHub, changes to the repository folder, launches the Multi-GPU training pipeline using HuggingFace Accelerate, and packs the outputs into a downloadable zip file.

In [ ]:
# 1. Git Repository Configuration & Clone
import os
import shutil

# ── Cấu hình Git Repo ──
GIT_USERNAME = "JustinYuanZe"
GIT_REPO = "TF_Biding_Project"
GIT_TOKEN = ""             # Điền GitHub Personal Access Token nếu là Private Repo, để trống nếu Public

# Xây dựng URL clone
if GIT_TOKEN:
    REPO_URL = f"https://{GIT_TOKEN}@github.com/{GIT_USERNAME}/{GIT_REPO}.git"
else:
    REPO_URL = f"https://github.com/{GIT_USERNAME}/{GIT_REPO}.git"

# Dọn dẹp thư mục cũ nếu có để tránh xung đột
if os.path.exists(GIT_REPO):
    shutil.rmtree(GIT_REPO)
    print(f"🧹 Removed existing folder: {GIT_REPO}")

# Thực hiện clone repository từ GitHub
print(f"🚀 Cloning repository from {GIT_USERNAME}/{GIT_REPO}...")
exit_code = os.system(f"git clone {REPO_URL}")

if exit_code == 0:
    print("✅ Clone successful!")
else:
    print("❌ ERROR: Git clone failed. Please check your username, repo name, or token.")


In [ ]:
# 2. Change directory and launch training
%cd TF_Biding_Project

import os
import subprocess
import torch

n_gpus = torch.cuda.device_count()
print(f'Detected {n_gpus} GPUs.')

cmd = ['accelerate', 'launch', '--mixed_precision=no']
if n_gpus > 1:
    cmd.extend(['--multi_gpu', f'--num_processes={n_gpus}'])
else:
    cmd.extend(['--num_processes=1'])

cmd.append('notebooks/20_hierarchical_kan_kaggle.py')

print(f'Running command: {" ".join(cmd)}')
# Run subprocess natively without redirecting stdout/stderr so Jupyter handles progress bars and Unicode natively
subprocess.run(cmd, check=True)


In [ ]:
# 3. Zip outputs and copy to parent directory for easy download
import os
import shutil
from IPython.display import FileLink

output_dir = "outputs_hierarchical_kan"
zip_name = "outputs_hierarchical_kan"

if os.path.exists(output_dir):
    print(f"📦 Zipping {output_dir}...")
    shutil.make_archive(zip_name, 'zip', output_dir)
    print(f"✅ Created {zip_name}.zip successfully!")
    
    # Copy to parent directory so it is visible in the root file explorer of Kaggle
    parent_zip_path = os.path.join("..", f"{zip_name}.zip")
    shutil.copy(f"{zip_name}.zip", parent_zip_path)
    print(f"💾 Copied zip to parent folder: {parent_zip_path}")
    
    # Display download link
    display(FileLink(parent_zip_path))
else:
    print(f"❌ Output directory {output_dir} not found. Check if training generated outputs.")
